# Fresh40 qualification: five cells
Use an A10040GB runtime. Cell1 uploads this complete package ZIP; cells2–5 install, freeze, score all models, and return results. Keep one runtime per Drive run folder. Read README before beginning. This is a development qualification, not a confirmatory final study.

In [ ]:
from pathlib import Path
import sys, subprocess, os, json, hashlib, zipfile, shutil
from google.colab import drive, files

drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/CSRA_Fresh40_runs/run1')
if (RUN / 'frozen_package').exists():
    ROOT = RUN / 'frozen_package'
    print('Resuming existing package:', ROOT)
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly one complete CSRA_Fresh40_Colab_Package.zip.')
    name, payload = next(iter(uploaded.items()))
    destination = Path('/content/CSRA_Fresh40_upload')
    destination.mkdir(exist_ok=True)
    import io
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for info in archive.infolist():
            target = (destination / info.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise RuntimeError('Unsafe archive path.')
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise RuntimeError('Archive symlinks are not allowed.')
        archive.extractall(destination)
    ROOT = destination / 'CSRA_Fresh40'
    if not (ROOT / 'protocol/package_lock.json').exists():
        raise RuntimeError('This is not the new Fresh40 package ZIP.')
lock = json.loads((ROOT / 'protocol/package_lock.json').read_text())
for name, expected in lock['files'].items():
    if hashlib.sha256((ROOT / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError('Changed package file: ' + name)
print('Package files verified. Run directory:', RUN)


In [ ]:
def visible(command):
    process = subprocess.Popen(list(map(str,command)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try: process.wait(timeout=20)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
    if result:
        raise RuntimeError(f'Step failed with exit code {result}; see full output above.')

# No notebook scientific imports: pinned work executes in fresh subprocesses.
visible([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'])
visible([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0+cu128', '--index-url', 'https://download.pytorch.org/whl/cu128'])
visible([sys.executable, '-m', 'pip', 'install', '-r', ROOT/'requirements-scoring.txt', '-r', ROOT/'requirements-analysis.txt'])
visible([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'])
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass

def stage(name, *extra):
    visible([sys.executable, '-u', ROOT/'code/run_fresh40.py', name, '--run-dir', RUN, *extra])
stage('validate')
stage('init')
ROOT = RUN / 'frozen_package'
print('Installation and initialization complete.')


Cell3 completes all preparation before any fresh scoring. It may spend several minutes fitting CPU models. On resume it verifies and reuses the freeze.

In [ ]:
stage('prepare')
print('All predictors and the observer selection are frozen. Continue to Cell 4.')


In [ ]:
for model in ('granite', 'phi', 'qwen'):
    print('\nTARGET ANSWERS:', model, flush=True)
    stage('answers', '--model', model)
stage('targets')
for model in ('granite', 'phi', 'qwen', 'mistral'):
    print('\nSOURCE REPORTS:', model, flush=True)
    stage('reports', '--model', model)
print('All 17,280 scoring jobs completed and checked.')


In [ ]:
stage('analyze')
RESULT_ZIP = RUN.parent / (RUN.name + '_CSRA_Fresh40_results.zip')
stage('export', '--output', RESULT_ZIP)
files.download(str(RESULT_ZIP))
print('Return the downloaded Fresh40 results ZIP for assessment.')
